# Matemáticas de la Inteligencia Artificial
## Sesión 1 — El perceptrón: aprender una frontera a partir de ejemplos

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CuentosCuanticos/matematicas-ia/blob/main/01_perceptron/laboratorio.ipynb)

**Objetivo.** Construir desde cero la máquina mínima de la sesión:

`datos → puntuación → predicción → error → actualización de parámetros`.

Usaremos solo **NumPy** y **Matplotlib**. No usamos una clase `Perceptron` ya hecha.

### Diccionario matemática ↔ código
- x: un ejemplo con dos características.
- X: matriz que contiene todos los ejemplos.
- y: etiqueta correcta, 0 o 1.
- w: vector de pesos.
- b: sesgo.
- z = w^T x + b: puntuación.
- y_hat: predicción.
- eta: tasa de aprendizaje.

Trabaja de arriba abajo. Las zonas `TODO` son las que debes completar.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)

n = 25
clase_0 = np.random.normal(loc=(-2.0, -1.5), scale=0.55, size=(n, 2))
clase_1 = np.random.normal(loc=( 2.0,  1.5), scale=0.55, size=(n, 2))
X = np.vstack([clase_0, clase_1])
y = np.concatenate([np.zeros(n, dtype=int), np.ones(n, dtype=int)])

print("X:", X.shape, "y:", y.shape)

plt.figure(figsize=(7, 6))
plt.scatter(X[y == 0, 0], X[y == 0, 1], label="Clase 0")
plt.scatter(X[y == 1, 0], X[y == 1, 1], label="Clase 1")
plt.xlabel("x1"); plt.ylabel("x2")
plt.title("Conjunto de entrenamiento")
plt.grid(alpha=.25); plt.legend(); plt.show()

## 1. Del número a la decisión

El perceptrón calcula primero una puntuación:

`z = w^T x + b`

y después aplica una función escalón: devuelve 1 cuando `z >= 0` y 0 cuando `z < 0`.

El producto `w^T x` se implementa en NumPy como `w @ x`. Por tanto, cada predicción contiene exactamente dos pasos: calcular `z` y decidir según su signo.

In [ ]:
def escalon(z):
    # TODO: devuelve 1 si z >= 0 y 0 en caso contrario.
    return ...

def predecir(x, w, b):
    # TODO: calcula z = w^T x + b.
    z = ...
    # TODO: usa la función escalón.
    return ...

w_prueba = np.array([0.6, -0.2])
b_prueba = 0.1

print("Predicción de prueba:", predecir(X[0], w_prueba, b_prueba))
print("Etiqueta real:", y[0])

## 2. El momento en que aparece el aprendizaje

Comparamos la etiqueta correcta `y` con la predicción `y_hat`. Definimos

`error = y - y_hat`.

La regla del perceptrón es:

`w ← w + eta * error * x`

`b ← b + eta * error`

Si el modelo acierta, `error = 0` y no cambia nada. Si se equivoca, el propio ejemplo indica en qué sentido deben modificarse los parámetros.

Primero implementamos **una sola corrección** y después construiremos el entrenamiento completo.

In [ ]:
eta = 0.2
w0 = np.zeros(2)
b0 = 0.0

x_i = X[0]
y_i = y[0]
y_hat_i = predecir(x_i, w0, b0)
error_i = y_i - y_hat_i

# TODO: traduce literalmente las dos ecuaciones de actualización.
w1 = ...
b1 = ...

print("y =", y_i, "y_hat =", y_hat_i, "error =", error_i)
print("antes:", w0, b0)
print("después:", w1, b1)

## 3. Una época y el entrenamiento completo

Una **época** es un recorrido completo por el conjunto de entrenamiento. En cada ejemplo:

1. predecimos;
2. calculamos `error = y - y_hat`;
3. si hay error, actualizamos `w` y `b`;
4. pasamos al siguiente ejemplo.

Repetimos épocas hasta que una termine sin errores o hasta alcanzar `max_epocas`.

Además guardaremos `historia` para poder ver cómo se mueve la frontera.

In [ ]:
def entrenar_perceptron(X, y, eta=0.2, max_epocas=50):
    w = np.zeros(X.shape[1])
    b = 0.0
    errores_por_epoca = []
    historia = [(w.copy(), b)]

    for epoca in range(max_epocas):
        errores = 0

        for x_i, y_i in zip(X, y):
            y_hat_i = predecir(x_i, w, b)
            error_i = y_i - y_hat_i

            if error_i != 0:
                # TODO: actualiza w y b.
                w = ...
                b = ...
                errores += 1
                historia.append((w.copy(), b))

        errores_por_epoca.append(errores)

        if errores == 0:
            break

    return w, b, errores_por_epoca, historia

w_final, b_final, errores, historia = entrenar_perceptron(X, y)

print("w final =", w_final)
print("b final =", b_final)
print("errores por época =", errores)

## 4. Hacemos visible la frontera

La frontera en dos dimensiones satisface:

`w1*x1 + w2*x2 + b = 0`.

Si `w2` no es cero, podemos despejar `x2` y dibujar la recta. Modifica `paso` para ver distintas etapas del aprendizaje.

In [ ]:
def dibujar_frontera(X, y, w, b, titulo):
    plt.figure(figsize=(7, 6))
    plt.scatter(X[y == 0, 0], X[y == 0, 1], label="Clase 0")
    plt.scatter(X[y == 1, 0], X[y == 1, 1], label="Clase 1")

    xs = np.linspace(X[:,0].min()-1, X[:,0].max()+1, 200)

    if abs(w[1]) > 1e-12:
        ys = -(w[0]*xs + b)/w[1]
        plt.plot(xs, ys, linewidth=2, label="Frontera")
    elif abs(w[0]) > 1e-12:
        plt.axvline(-b/w[0], linewidth=2, label="Frontera")

    plt.xlabel("x1"); plt.ylabel("x2")
    plt.title(titulo); plt.grid(alpha=.25); plt.legend(); plt.show()

dibujar_frontera(X, y, w_final, b_final, "Perceptrón entrenado")

paso = -1
w_paso, b_paso = historia[paso]
dibujar_frontera(X, y, w_paso, b_paso, f"Frontera: paso {paso}")

## 5. Dos experimentos

### A. Cambia `eta`
Prueba `eta = 0.05`, `0.2`, `1.0` y `5.0`. ¿Cambia el número de errores? ¿Cambian los valores de los parámetros?

### B. Cambia la representación
Haz una copia de `X` y multiplica la primera columna por 10. Entrena otra vez. La información conceptual puede ser la misma, pero la regla de actualización usa directamente las coordenadas. Esto anticipa la pregunta de la sesión 2: **la representación y su geometría importan**.

### Preguntas finales
1. ¿Qué parte del programa es el modelo y qué parte es el algoritmo de aprendizaje?
2. ¿Por qué no se actualizan los parámetros cuando `y == y_hat`?
3. ¿Qué significa que una época termine con cero errores?
4. ¿Garantiza eso clasificar correctamente cualquier dato nuevo?
5. Explica con tus palabras qué significa “aprender” en este experimento.

### Hito acumulativo
Has construido el primer ciclo completo del curso:

`datos → predicción → error → actualización de parámetros`.

Guarda una copia personal en Colab. La versión de GitHub permanece como referencia canónica.